In [1]:
from torchtext.data.utils import get_tokenizer
from torchtext.vocab import build_vocab_from_iterator
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "C:\Anaconda\envs\study_env2\Lib\runpy.py", line 198, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "C:\Anaconda\envs\study_env2\Lib\runpy.py", line 88, in _run_code
    exec(code, run_globals)
  File "C:\Anaconda\envs\study_env2\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "C:\Anaconda\envs\study_env2\Lib\site-packages\traitlets\config\application.py", line 1082, in launch_instance
    app.start()
  File "C:\Anaconda\envs\study_en

# Tokenizer

In [2]:
tokenizer = get_tokenizer("basic_english")

In [3]:
sentence = 'Tôi đi học AI'

In [4]:
tokenizer(sentence)

['tôi', 'đi', 'học', 'ai']

# built vocab

In [5]:
list_sentence = ['Tôi đi học AI', 'Tôi không biết rõ AI là như thế nào']

In [6]:
# Sử lý từng mẩu sample trong list_sentence khi được gọi
def yield_tokens(data):
    for sentence in data:
        yield tokenizer(sentence)

In [7]:
# Định nghĩa kích thước của vocab
vocab_size = 8
vocab = build_vocab_from_iterator(
    iterator= yield_tokens(list_sentence),
    specials=['<pad>', '<unk>'],
    max_tokens=vocab_size
)

# set các từ không xuất hiện trong từ điển có giá trị index là giá trị của từ <unk>
vocab.set_default_index(vocab['<unk>'])
vocab.get_stoi()

{'không': 6,
 '<pad>': 0,
 '<unk>': 1,
 'tôi': 3,
 'ai': 2,
 'biết': 4,
 'học': 5,
 'là': 7}

# index presentation

In [8]:
for sentence in list_sentence:
    sentence_index = vocab(tokenizer(sentence))
    print(sentence_index)

[3, 1, 5, 2]
[3, 6, 4, 1, 2, 7, 1, 1, 1]


# Xử lý câu ngắn câu dài -> sequence length cố định trước khi đưa vào mạng

In [9]:
class TextDataset(Dataset):
    def __init__(self,data, max_length):
        self.data = []
        for sentence in data:
            sentence_index = vocab(tokenizer(sentence))
            #  num padding
            num_pad = max_length - len(sentence_index)
            sentence_index = sentence_index[:max_length] + [vocab['<pad>']]*num_pad
            self.data.append(sentence_index)
    def __len__(self):
        return len(self.data)
    def __getitem__(self,index):
        return torch.tensor(self.data[index])

In [10]:
list_sentence

['Tôi đi học AI', 'Tôi không biết rõ AI là như thế nào']

In [11]:
dataset = TextDataset(list_sentence, max_length=5)

In [12]:
print(dataset[0])
print(dataset[1])

tensor([3, 1, 5, 2, 0])
tensor([3, 6, 4, 1, 2])


In [13]:
dataloader = DataLoader(dataset, batch_size=2)

# Layer Embedding

In [14]:
vocab_size

8

In [15]:
embedder = nn.Embedding(num_embeddings=vocab_size, embedding_dim=4)
embedder.weight

Parameter containing:
tensor([[-1.3055, -1.1539, -0.1480, -0.2062],
        [-0.7689, -1.2053, -0.6433, -0.2907],
        [-0.2240,  1.8277,  0.4430, -1.4976],
        [-0.6256,  1.0850,  1.0181,  1.0892],
        [ 1.1122,  0.7741,  0.0793, -1.2798],
        [-4.2095,  0.6189, -1.8583,  0.2627],
        [ 1.5483,  0.1362,  0.7182,  0.1463],
        [ 0.6085,  0.2655, -0.1166, -0.0852]], requires_grad=True)

In [16]:
for X in dataloader:
    matrix_embedd = embedder(X)
matrix_embedd.shape

torch.Size([2, 5, 4])

# RNN layer

In [17]:
rnn_layer =nn.RNN(
    input_size = 4,# embedding dim
    hidden_size = 3, # hidden_state dim
    batch_first = True, # quy định đầu vào và đầu ra có dạng (batch_size, sequence_length, embedding_dim)
)
rnn_layer

RNN(4, 3, batch_first=True)

In [18]:
print(rnn_layer.weight_hh_l0)
print(rnn_layer.weight_ih_l0)

Parameter containing:
tensor([[-0.2410, -0.2814,  0.2499],
        [ 0.3427, -0.0885, -0.1483],
        [-0.1770, -0.4876, -0.1995]], requires_grad=True)
Parameter containing:
tensor([[-0.5660, -0.1999, -0.4611, -0.5641],
        [-0.2547,  0.4063, -0.4812,  0.0321],
        [-0.0098,  0.0267,  0.2523,  0.1895]], requires_grad=True)


In [19]:
print(rnn_layer.bias_hh_l0)
print(rnn_layer.bias_ih_l0)

Parameter containing:
tensor([ 0.2019, -0.0910, -0.3421], requires_grad=True)
Parameter containing:
tensor([ 0.2996,  0.1900, -0.0092], requires_grad=True)


In [20]:
 output, hn= rnn_layer(matrix_embedd)

In [21]:
output.shape

torch.Size([2, 5, 3])

In [22]:
hn

tensor([[[ 0.8329,  0.1502, -0.6048],
         [ 0.4257,  0.7629, -0.5877]]], grad_fn=<StackBackward0>)

In [23]:
hn[-1]

tensor([[ 0.8329,  0.1502, -0.6048],
        [ 0.4257,  0.7629, -0.5877]], grad_fn=<SelectBackward0>)